# AI-Assisted Parallel Image Processing — Colab setup

Trước khi chạy, chọn **Runtime → Change runtime type → GPU**. Notebook sẽ clone repository, cài dependency, tải dữ liệu có kiểm tra checksum, tạo bộ benchmark và build bản Release bằng OpenMP/CUDA.

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
REPOSITORY = 'https://github.com/Chicken20145/ai-assisted-parallel-image-processing.git'
GIT_REF = 'feature/core-api-cpu'  # Đổi thành main sau khi PR được merge.

%cd /content
!test -d ai-assisted-parallel-image-processing || git clone --branch {GIT_REF} {REPOSITORY}
%cd /content/ai-assisted-parallel-image-processing
!git fetch origin {GIT_REF}
!git switch {GIT_REF}
!git pull --ff-only origin {GIT_REF}
!bash scripts/setup_colab.sh

## Cập nhật code khi A/B/C làm song song

Sau khi thành viên A push commit mới lên cùng branch, chạy cell dưới để lấy code mới, build tăng dần và test lại. Không cần tải lại dataset hoặc cài lại package trong cùng runtime.

In [ ]:
%cd /content/ai-assisted-parallel-image-processing
!git pull --ff-only origin {GIT_REF}
!bash scripts/build_colab.sh
!bash scripts/test_colab.sh

## Lưu kết quả benchmark lên Drive (tùy chọn)

Chỉ mount Drive khi cần lưu kết quả. Nên benchmark trên ổ đĩa cục bộ `/content` rồi mới chép CSV/biểu đồ sang Drive để I/O mạng không làm sai lệch thời gian.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil

source = Path('/content/ai-assisted-parallel-image-processing/benchmarks/results')
destination = Path('/content/drive/MyDrive/parallel-image-processing/results')
if source.exists():
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, destination, dirs_exist_ok=True)
    print(f'Copied results to {destination}')
else:
    print('No benchmark results yet.')